# 04 Limpieza — separación artículo–autor–afiliación con reglas explícitas por fuente

Esta libreta procesa el archivo canónico integrado y genera **un solo archivo final** con una fila por autor.

Características:

- Usa el archivo `integrado_preliminar_canonico.csv`.
- Detecta la fuente/editorial desde el prefijo de `indice`.
- Aplica reglas explícitas para `ACM`, `EBSCO`, `EV`, `IEEE`, `PROQUEST`, `SD`, `SCOPUS` y `WOS`.
- Guarda sólo `integrado_articulo_autor_afiliacion_unificado.csv`.
- Conserva exactamente las 14 columnas canónicas.

La salida se guarda en:

`C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\00_Separacion_Autor_Afiliacion`


## Configuración de rutas, columnas y funciones generales

In [1]:
from __future__ import annotations

import csv
import re
import unicodedata
import difflib
import math
import json
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple, Optional

CANONICAL_COLUMNS = [
    "indice", "Titulo", "Año", "Autor_norm", "Afiliacion1", "Afiliacion2",
    "ISBN", "ISSN", "Doi", "URL", "Area", "Subarea", "Keywords", "Abstract"
]

INPUT_INTEGRADO = Path(r"C:\Users\hazar\Desktop\PROYECTO\02_modelo canonico\03_union\integrado_preliminar_canonico.csv")
OUTPUT_DIR = Path(r"C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\00_Separacion_Autor_Afiliacion")


if not INPUT_INTEGRADO.exists() and Path("/mnt/data/integrado_preliminar_canonico.csv").exists():
    INPUT_INTEGRADO = Path("/mnt/data/integrado_preliminar_canonico.csv")
    OUTPUT_DIR = Path("/mnt/data/00_Separacion_Autor_Afiliacion_reglas_explicitas")

OUTPUT_FINAL = OUTPUT_DIR / "integrado_articulo_autor_afiliacion_unificado.csv"

SOURCE_ORDER = [
    "ACM", "EBSCO", "ENGINEERING_VILLAGE", "IEEE", "PROQUEST", "SCIENCEDIRECT", "SCOPUS", "WOS"
]

# -----------------------------------------------------------------------------
# Utilidades generales
# -----------------------------------------------------------------------------

def read_csv(path: Path) -> List[Dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames != CANONICAL_COLUMNS:
            raise ValueError(f"Columnas no canónicas en {path}. Encontradas={reader.fieldnames}")
        return [{k: (v or "") for k, v in row.items()} for row in reader]


def write_csv(path: Path, rows: List[Dict[str, Any]], columns: List[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow({col: row.get(col, "") for col in columns})


def detect_source_from_indice(indice: str) -> str:
    idx = str(indice or "").strip().upper()
    prefix = idx.split("_", 1)[0]
    if prefix == "ACM":
        return "ACM"
    if prefix == "EBSCO":
        return "EBSCO"
    if prefix in {"EV", "ENGINEERING", "ENGINEERINGVILLAGE", "ENGINEERING-VILLAGE"}:
        return "ENGINEERING_VILLAGE"
    if prefix == "IEEE":
        return "IEEE"
    if prefix == "PROQUEST":
        return "PROQUEST"
    if prefix in {"SD", "SCIENCEDIRECT", "SCIENCE"}:
        return "SCIENCEDIRECT"
    if prefix == "SCOPUS":
        return "SCOPUS"
    if prefix in {"WOS", "WEB", "WEBOFSCIENCE", "WEB-OF-SCIENCE"}:
        return "WOS"
    raise ValueError(f"No se pudo detectar fuente para indice={indice!r}")


def group_integrated_rows(rows: List[Dict[str, str]]) -> Dict[str, List[Tuple[int, Dict[str, str]]]]:
    grouped: Dict[str, List[Tuple[int, Dict[str, str]]]] = {source: [] for source in SOURCE_ORDER}
    for global_article_idx, row in enumerate(rows, start=1):
        source = detect_source_from_indice(row.get("indice", ""))
        grouped[source].append((global_article_idx, row))
    return grouped


def strip_accents_general(text: str) -> str:
    text = "" if text is None else str(text)
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))

## Reglas ACM

In [2]:
# -----------------------------------------------------------------------------
# ACM
# -----------------------------------------------------------------------------

def acm_strip_accents(text: str) -> str:
    text = "" if text is None else str(text)
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))


def acm_norm_text(text: str) -> str:
    text = acm_strip_accents(text).lower()
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def acm_norm_name(text: str) -> str:
    text = acm_strip_accents(text).lower()
    text = re.sub(r"[.,;:()\[\]{}]", " ", text)
    text = re.sub(r"[-_/]+", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def acm_invert_author_norm(author: str) -> str:
    author = (author or "").strip()
    if "," in author:
        last, rest = author.split(",", 1)
        return f"{rest.strip()} {last.strip()}".strip()
    return author


def acm_split_authors(author_field: str) -> List[str]:
    return [a.strip() for a in (author_field or "").split(";") if a.strip()]


def acm_is_unam_affiliation(text: str) -> bool:
    n = acm_norm_text(text)
    patterns = [r"\bunam\b", r"universidad nacional autonoma de mexico", r"national autonomous university of mexico", r"u n a m"]
    return any(re.search(p, n) for p in patterns)


def acm_unique_preserve_order(items: Iterable[str]) -> List[str]:
    out: List[str] = []
    seen = set()
    for item in items:
        item = re.sub(r"\s+", " ", (item or "").strip().strip(";|"))
        if not item:
            continue
        key = acm_norm_text(item)
        if key not in seen:
            seen.add(key)
            out.append(item)
    return out


def acm_split_embedded_affiliations(aff_text: str) -> List[str]:
    if not aff_text:
        return []
    parts = [p.strip() for p in re.split(r"\s*;\s*", aff_text) if p.strip()]
    return parts or [aff_text.strip()]


def acm_parse_colon_entries(aff_text: str) -> List[Tuple[str, str]]:
    text = re.sub(r"\s+", " ", aff_text or "").strip()
    if not text or ":" not in text:
        return []
    pattern = re.compile(r"(?:^|[;|])\s*([^:;|]{2,140}?)\s*:\s*")
    matches = list(pattern.finditer(text))
    entries: List[Tuple[str, str]] = []
    for i, match in enumerate(matches):
        name = match.group(1).strip()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        aff = text[start:end].strip().strip(";|").strip()
        if name and aff:
            for part in acm_split_embedded_affiliations(aff):
                entries.append((name, part))
    return entries


def acm_parse_marker_entries(aff_text: str) -> Tuple[List[Tuple[str, List[str]]], Dict[str, List[str]]]:
    text = re.sub(r"\s+", " ", aff_text or "").strip()
    if "|" not in text or not re.search(r"\[\d+\]", text):
        return [], {}
    left, right = text.split("|", 1)
    author_markers: List[Tuple[str, List[str]]] = []
    for segment in re.split(r"\s*;\s*", left):
        segment = segment.strip()
        if not segment:
            continue
        m = re.match(r"^(.*?)\s*((?:\[\d+\]\s*)+)$", segment)
        if not m:
            continue
        name = m.group(1).strip()
        nums = re.findall(r"\[(\d+)\]", m.group(2))
        if name and nums:
            author_markers.append((name, nums))
    marker_map: Dict[str, List[str]] = defaultdict(list)
    def_matches = list(re.finditer(r"\[(\d+)\]\s*", right))
    for i, m in enumerate(def_matches):
        marker = m.group(1)
        start = m.end()
        end = def_matches[i + 1].start() if i + 1 < len(def_matches) else len(right)
        aff = right[start:end].strip().strip(";|").strip()
        for part in acm_split_embedded_affiliations(aff):
            marker_map[marker].append(part)
    return author_markers, dict(marker_map)


def acm_author_match_score(extracted_name: str, canonical_author: str) -> float:
    ext = acm_norm_name(extracted_name)
    can_display = acm_norm_name(acm_invert_author_norm(canonical_author))
    can_raw = acm_norm_name(canonical_author)
    if not ext or not can_display:
        return 0.0
    if ext == can_display:
        return 1.0
    score = max(SequenceMatcher(None, ext, can_display).ratio(), SequenceMatcher(None, ext, can_raw).ratio())
    e_tokens = ext.split()
    c_tokens = can_display.split()
    if e_tokens and c_tokens:
        e_set, c_set = set(e_tokens), set(c_tokens)
        overlap = len(e_set & c_set)
        overlap_score = overlap / max(len(e_set), len(c_set))
        score = max(score, overlap_score)
        if e_set.issubset(c_set) and len(e_set) >= 2:
            score = max(score, 0.92)
        if c_set.issubset(e_set) and len(c_set) >= 2:
            score = max(score, 0.92)
        if e_tokens[0] == c_tokens[0] and e_tokens[-1] == c_tokens[-1]:
            score = max(score, 0.88)
    return score


def acm_match_name_to_author_index(extracted_name: str, authors: List[str]) -> Tuple[Optional[int], float]:
    scores = [(i, acm_author_match_score(extracted_name, author)) for i, author in enumerate(authors)]
    if not scores:
        return None, 0.0
    scores.sort(key=lambda x: x[1], reverse=True)
    best_i, best_score = scores[0]
    second_score = scores[1][1] if len(scores) > 1 else 0.0
    if best_score >= 0.78 and (best_score - second_score >= 0.06 or best_score >= 0.92):
        return best_i, best_score
    return None, best_score


def acm_choose_two_affiliations(affs: List[str]) -> Tuple[str, str]:
    affs = acm_unique_preserve_order(affs)
    if len(affs) <= 2:
        return (affs[0] if len(affs) >= 1 else "", affs[1] if len(affs) >= 2 else "")
    unam = [a for a in affs if acm_is_unam_affiliation(a)]
    if unam:
        kept = unam[:2]
        return kept[0], kept[1] if len(kept) > 1 else ""
    return affs[0], affs[1]


def process_acm_rows(rows: List[Dict[str, str]], global_ids: List[int]) -> List[Dict[str, str]]:
    final_rows: List[Dict[str, str]] = []
    for row, article_id in zip(rows, global_ids):
        authors = acm_split_authors(row.get("Autor_norm", ""))
        n_authors = len(authors)
        aff_raw = " | ".join([v for v in [row.get("Afiliacion1", ""), row.get("Afiliacion2", "")] if v])
        assigned: Dict[int, List[str]] = defaultdict(list)
        marker_author_entries, marker_map = acm_parse_marker_entries(aff_raw)
        colon_entries = acm_parse_colon_entries(aff_raw)

        if marker_author_entries and marker_map:
            for extracted_name, markers in marker_author_entries:
                idx, score = acm_match_name_to_author_index(extracted_name, authors)
                if idx is None:
                    continue
                for marker in markers:
                    for aff in marker_map.get(marker, []):
                        assigned[idx].append(aff)

        if colon_entries:
            for extracted_name, aff in colon_entries:
                idx, score = acm_match_name_to_author_index(extracted_name, authors)
                if idx is None:
                    continue
                assigned[idx].append(aff)

        if n_authors == 1 and not assigned and aff_raw.strip():
            fallback_affs = [a for _, a in colon_entries] if colon_entries else acm_split_embedded_affiliations(aff_raw)
            assigned[0].extend(fallback_affs)

        for pos, author in enumerate(authors, start=1):
            idx = pos - 1
            aff1, aff2 = acm_choose_two_affiliations(assigned.get(idx, []))
            final = {col: row.get(col, "") for col in CANONICAL_COLUMNS}
            final["indice"] = str(article_id)
            final["Autor_norm"] = author
            final["Afiliacion1"] = aff1
            final["Afiliacion2"] = aff2
            final_rows.append(final)
    return final_rows

## Reglas EBSCO

In [3]:
# -----------------------------------------------------------------------------
# EBSCO
# -----------------------------------------------------------------------------

def ebsco_norm_text(value: object) -> str:
    text = "" if value is None else str(value)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower().replace("&", " and ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def ebsco_split_authors(text: str) -> List[str]:
    return [x.strip() for x in re.split(r"\s*;\s*", text or "") if x.strip()]


def ebsco_split_blocks(text: str) -> List[str]:
    return [x.strip() for x in re.split(r"\s*\|\s*", text or "") if x.strip()]


def ebsco_split_block_author_affiliation(block: str) -> Tuple[str, str, bool]:
    parts = re.split(r"\s*[—–]\s*|\s+-\s+", block or "", maxsplit=1)
    if len(parts) == 2:
        return parts[0].strip(), parts[1].strip(), True
    return "", (block or "").strip(), False


def ebsco_is_unam(text: str) -> bool:
    x = ebsco_norm_text(text)
    if not x:
        return False
    patterns = [
        r"\bunam\b", r"universidad nacional autonoma de mexico", r"national autonomous university of mexico",
        r"univ nacl autonoma mex", r"univ nacional autonoma de mexico", r"uaem unam", r"unam uaem",
        r"ciudad universitaria", r"instituto de investigaciones en matematicas aplicadas y en sistemas", r"iimas",
        r"facultad de estudios superiores", r"instituto de ingenieria universidad nacional",
        r"facultad de ingenieria universidad nacional", r"instituto de geociencias universidad nacional",
        r"instituto de quimica universidad nacional", r"facultad de ciencias universidad nacional",
        r"centro de ciencias de la complejidad", r"centro de ciencias matematicas", r"lavis universidad nacional",
    ]
    return any(re.search(p, x) for p in patterns)


def ebsco_is_note_not_affiliation(text: str) -> bool:
    x = ebsco_norm_text(text)
    return any(p in x for p in [
        "afiliacion exacta no exportada", "affiliation not exported", "affiliation not available", "sin afiliacion exacta"
    ])


def ebsco_split_affiliations_for_author(affiliation_text: str) -> List[str]:
    parts = [p.strip() for p in re.split(r"\s*;\s*", affiliation_text or "") if p.strip()]
    return [p for p in parts if not ebsco_is_note_not_affiliation(p)]


def ebsco_choose_affiliation_columns(parts: List[str]) -> Tuple[str, str]:
    if not parts:
        return "", ""
    if len(parts) == 1:
        return parts[0], ""
    if len(parts) == 2:
        return parts[0], parts[1]
    unam_parts = [p for p in parts if ebsco_is_unam(p)]
    if unam_parts:
        return unam_parts[0], (unam_parts[1] if len(unam_parts) > 1 else "")
    return parts[0], parts[1]


def process_ebsco_rows(rows: List[Dict[str, str]], global_ids: List[int]) -> List[Dict[str, str]]:
    final_rows: List[Dict[str, str]] = []
    for row, article_id in zip(rows, global_ids):
        authors = ebsco_split_authors(row.get("Autor_norm", ""))
        blocks = ebsco_split_blocks(row.get("Afiliacion1", ""))
        for pos, author in enumerate(authors, start=1):
            block = blocks[pos - 1] if pos <= len(blocks) else ""
            block_author, block_aff, has_sep = ebsco_split_block_author_affiliation(block)
            parts = ebsco_split_affiliations_for_author(block_aff)
            aff1, aff2 = ebsco_choose_affiliation_columns(parts)
            out = {col: row.get(col, "") for col in CANONICAL_COLUMNS}
            out["indice"] = str(article_id)
            out["Autor_norm"] = author
            out["Afiliacion1"] = aff1
            out["Afiliacion2"] = aff2
            final_rows.append(out)
    return final_rows

## Reglas Engineering Village / EV

In [4]:
# -----------------------------------------------------------------------------
# Engineering Village / EV
# -----------------------------------------------------------------------------

EV_PROHIBITED_AFF_PATTERNS = [
    "afiliacion externa", "externa no unam", "externo no unam", "no unam en la fuente consultada", "no unam",
    "fuente consultada", "afiliacion no recuperada", "no recuperada", "no disponible", "sin afiliacion",
    "sin informacion de afiliacion", "exacta no exportada", "no exportada", "external affiliation", "external no unam",
]
EV_STOPWORDS = {
    "the", "and", "for", "with", "from", "into", "that", "this", "their", "de", "del", "la", "las",
    "los", "el", "en", "y", "a", "of", "in", "on", "to", "para", "por", "con",
    "university", "universidad", "national", "nacional", "autonomous", "autonoma", "autonomo",
    "department", "departamento", "dept", "faculty", "facultad", "school", "college", "division",
    "institute", "instituto", "center", "centro", "research", "investigacion", "investigaciones",
    "laboratory", "laboratorio", "science", "sciences", "ciencia", "ciencias", "engineering", "ingenieria",
    "postgraduate", "postgrado", "posgrado", "graduate", "program", "programa", "studies", "estudios",
    "mexico", "mexican", "mexicana", "city", "ciudad", "cdmx", "united", "states", "state", "france",
    "spain", "chile", "japan", "germany", "italy", "argentina", "canada", "brazil", "netherlands",
    "china", "korea", "south", "north", "kingdom", "uk", "usa", "ma", "ca", "pa", "ny",
}


def ev_clean_spaces(value: Any) -> str:
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\r", " ").replace("\n", " ")).strip()


def ev_norm_text(value: Any) -> str:
    value = ev_clean_spaces(value)
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = value.lower()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def ev_tokens(value: Any) -> set:
    n = ev_norm_text(value)
    return {tok for tok in n.split() if len(tok) >= 3 and tok not in EV_STOPWORDS}


def ev_is_prohibited_affiliation(value: Any) -> bool:
    n = ev_norm_text(value)
    if not n:
        return True
    return any(pat in n for pat in EV_PROHIBITED_AFF_PATTERNS)


def ev_is_unam_affiliation(value: Any) -> bool:
    n = ev_norm_text(value)
    if not n or ev_is_prohibited_affiliation(value):
        return False
    direct_patterns = [
        r"\bunam\b", "universidad nacional autonoma de mexico", "universidad nacional autonoma mexico",
        "national autonomous university of mexico", "national autonomous university mexico",
    ]
    if any(re.search(p, n) if p.startswith(r"\b") else p in n for p in direct_patterns):
        return True
    strong_dependencies = [
        "iimas", "instituto de investigaciones en matematicas aplicadas y en sistemas", "instituto de matematicas",
        "instituto de astronomia", "instituto de ciencias nucleares", "facultad de estudios superiores aragon",
        "facultad de ciencias", "facultad de ingenieria", "facultad de psicologia", "instituto de neurobiologia",
    ]
    external_univ_markers = [
        "autonomous university of queretaro", "universidad autonoma de queretaro", "universidad de colima",
        "universidad de sonora", "universidad de guadalajara", "cinvestav", "ipn", "uam",
        "universidad politecnica", "universidad tecnica", "universidad carlos", "universidad de jaen",
        "kyoto university", "hokkaido university", "massachusetts institute", "texas a m", "rutgers",
    ]
    if any(dep in n for dep in strong_dependencies) and not any(ext in n for ext in external_univ_markers):
        return True
    return False


def ev_split_authors(author_cell: Any) -> List[str]:
    return [ev_clean_spaces(a) for a in str(author_cell or "").split(";") if ev_clean_spaces(a)]


def ev_parse_numbered_affiliations(aff1: Any, aff2: Any = "") -> List[Dict[str, str]]:
    text = ev_clean_spaces(" ".join([str(aff1 or ""), str(aff2 or "")]).strip())
    if not text:
        return []
    pattern = re.compile(r"\((\d+)\)\s*(.*?)(?=\s*\(\d+\)\s*|$)")
    matches = list(pattern.finditer(text))
    affs: List[Dict[str, str]] = []
    if matches:
        for m in matches:
            marker = m.group(1)
            aff_text = ev_clean_spaces(m.group(2)).strip(" ;")
            if aff_text and not ev_is_prohibited_affiliation(aff_text):
                affs.append({"marker": marker, "text": aff_text})
        return affs
    if text and not ev_is_prohibited_affiliation(text):
        return [{"marker": "", "text": text.strip(" ;")}]
    return []


def ev_dedupe_aff_texts(affs: Iterable[str]) -> List[str]:
    seen = set()
    out = []
    for aff in affs:
        aff = ev_clean_spaces(aff).strip(" ;")
        if not aff or ev_is_prohibited_affiliation(aff):
            continue
        key = ev_norm_text(aff)
        if key not in seen:
            seen.add(key)
            out.append(aff)
    return out


def ev_limit_two_affiliations(affs: List[str]) -> Tuple[str, str]:
    affs = ev_dedupe_aff_texts(affs)
    if len(affs) <= 2:
        return (affs[0] if len(affs) > 0 else "", affs[1] if len(affs) > 1 else "")
    unam_affs = [a for a in affs if ev_is_unam_affiliation(a)]
    if unam_affs:
        selected = unam_affs[:2]
        return (selected[0] if selected else "", selected[1] if len(selected) > 1 else "")
    selected = affs[:2]
    return selected[0], selected[1]


def ev_aff_similarity(known_aff: str, candidate_aff: str) -> float:
    if not known_aff or not candidate_aff:
        return 0.0
    nk = ev_norm_text(known_aff)
    nc = ev_norm_text(candidate_aff)
    if not nk or not nc:
        return 0.0
    if nk == nc:
        return 1.0
    if len(nk) >= 18 and len(nc) >= 18 and (nk in nc or nc in nk):
        return 0.98
    if ev_is_unam_affiliation(known_aff) and ev_is_unam_affiliation(candidate_aff):
        return 0.86
    tk, tc = ev_tokens(known_aff), ev_tokens(candidate_aff)
    if tk and tc:
        containment = len(tk & tc) / max(1, min(len(tk), len(tc)))
        jaccard = len(tk & tc) / max(1, len(tk | tc))
    else:
        containment = 0.0
        jaccard = 0.0
    ratio = SequenceMatcher(None, nk, nc).ratio()
    return max(containment * 0.85, jaccard * 0.70, ratio * 0.60)


def ev_best_history_match(author: str, candidate_affs: List[Dict[str, str]], author_history: Dict[str, List[str]]) -> List[str]:
    known = author_history.get(author, [])
    if not known or not candidate_affs:
        return []
    scored: List[Tuple[float, str]] = []
    for cand in candidate_affs:
        best = max((ev_aff_similarity(k, cand["text"]) for k in known), default=0.0)
        if best >= 0.72:
            scored.append((best, cand["text"]))
    if not scored:
        return []
    scored.sort(key=lambda x: x[0], reverse=True)
    top_score = scored[0][0]
    selected = [text for score, text in scored if score >= 0.90 or (score >= 0.82 and abs(score - top_score) <= 0.06)]
    if selected:
        return ev_dedupe_aff_texts(selected)
    return [scored[0][1]]


def ev_distribute_affiliations(authors: List[str], affs: List[Dict[str, str]], author_history: Dict[str, List[str]]) -> List[Dict[str, Any]]:
    n_auth = len(authors)
    n_aff = len(affs)
    assignments: List[Dict[str, Any]] = []
    if n_auth == 0:
        return assignments
    if n_aff == 0:
        for i, author in enumerate(authors, start=1):
            assignments.append({"author": author, "pos": i, "affs": []})
        return assignments
    if n_auth == 1:
        assignments.append({"author": authors[0], "pos": 1, "affs": [a["text"] for a in affs]})
        return assignments
    if n_aff == n_auth:
        for i, author in enumerate(authors):
            assignments.append({"author": author, "pos": i + 1, "affs": [affs[i]["text"]]})
        return assignments
    if n_aff == 1:
        for i, author in enumerate(authors, start=1):
            assignments.append({"author": author, "pos": i, "affs": [affs[0]["text"]]})
        return assignments
    preliminary: List[Optional[Dict[str, Any]]] = [None for _ in authors]
    for i, author in enumerate(authors):
        matched = ev_best_history_match(author, affs, author_history)
        if matched:
            preliminary[i] = {"author": author, "pos": i + 1, "affs": matched}
    if n_aff > n_auth:
        fallback_affs_by_author: List[List[Dict[str, str]]] = [[] for _ in authors]
        for i in range(n_auth - 1):
            fallback_affs_by_author[i].append(affs[i])
        fallback_affs_by_author[n_auth - 1].extend(affs[n_auth - 1:])
    else:
        fallback_affs_by_author = [[] for _ in authors]
        for i in range(n_auth):
            aff_idx = min(n_aff - 1, int(math.floor(i * n_aff / n_auth)))
            fallback_affs_by_author[i].append(affs[aff_idx])
    for i, author in enumerate(authors):
        if preliminary[i] is not None:
            assignments.append(preliminary[i])
        else:
            f_affs = fallback_affs_by_author[i]
            assignments.append({"author": author, "pos": i + 1, "affs": [a["text"] for a in f_affs]})
    return assignments


def ev_build_author_history(rows: List[Dict[str, str]]) -> Dict[str, List[str]]:
    history: Dict[str, List[str]] = defaultdict(list)
    for row in rows:
        authors = ev_split_authors(row.get("Autor_norm", ""))
        affs = ev_parse_numbered_affiliations(row.get("Afiliacion1", ""), row.get("Afiliacion2", ""))
        n_auth, n_aff = len(authors), len(affs)
        if not authors or not affs:
            continue
        if n_auth == 1:
            for aff in affs:
                history[authors[0]].append(aff["text"])
        elif n_aff == n_auth:
            for author, aff in zip(authors, affs):
                history[author].append(aff["text"])
        elif n_aff == 1:
            for author in authors:
                history[author].append(affs[0]["text"])
        elif n_aff > n_auth:
            for i, author in enumerate(authors):
                if i < n_auth - 1:
                    history[author].append(affs[i]["text"])
                else:
                    for aff in affs[i:]:
                        history[author].append(aff["text"])
    return {author: ev_dedupe_aff_texts(aff_list) for author, aff_list in history.items()}


def process_ev_rows(rows: List[Dict[str, str]], global_ids: List[int]) -> List[Dict[str, str]]:
    author_history = ev_build_author_history(rows)
    final_rows: List[Dict[str, str]] = []
    for row, article_id in zip(rows, global_ids):
        authors = ev_split_authors(row.get("Autor_norm", ""))
        affs = ev_parse_numbered_affiliations(row.get("Afiliacion1", ""), row.get("Afiliacion2", ""))
        assignments = ev_distribute_affiliations(authors, affs, author_history)
        for a in assignments:
            aff1, aff2 = ev_limit_two_affiliations(a["affs"])
            final = {col: row.get(col, "") for col in CANONICAL_COLUMNS}
            final["indice"] = str(article_id)
            final["Autor_norm"] = a["author"]
            final["Afiliacion1"] = aff1
            final["Afiliacion2"] = aff2
            final_rows.append(final)
    return final_rows

## Reglas IEEE

In [5]:
# -----------------------------------------------------------------------------
# IEEE
# -----------------------------------------------------------------------------

IEEE_PLACEHOLDER_PATTERNS = [
    r"afiliaci[oó]n\s+externa", r"no\s+unam", r"no\s+recuperad[ao]",
    r"affiliation\s+(?:not\s+)?(?:available|exported|recovered)", r"exacta\s+no\s+exportada", r"sin\s+afiliaci[oó]n",
]
IEEE_UNAM_PATTERNS = [
    r"\bunam\b", r"universidad\s+nacional\s+autonoma\s+de\s+mexico",
    r"universidad\s+nacional\s+autónoma\s+de\s+méxico", r"national\s+autonomous\s+university\s+of\s+mexico",
    r"national\s+autonomous\s+univ(?:ersity)?\s+of\s+mexico",
    r"univ\.?\s+nac\.?\s+aut[oó]noma\s+(?:de\s+)?m[eé]xico", r"facultad\s+de\s+estudios\s+superiores",
    r"\bfes\s+(?:aragon|aragón|acatlan|acatlán|cuautitlan|cuautitlán|iztacala|zaragoza)\b",
    r"instituto\s+de\s+investigaciones\s+en\s+matem[aá]ticas\s+aplicadas\s+y\s+en\s+sistemas",
    r"\biimas\b", r"centro\s+de\s+ciencias\s+de\s+la\s+complejidad", r"centro\s+de\s+ciencias\s+matem[aá]ticas",
]


def ieee_strip_accents(text: str) -> str:
    return "".join(char for char in unicodedata.normalize("NFD", str(text)) if unicodedata.category(char) != "Mn")


def ieee_norm_for_matching(text: str) -> str:
    text = ieee_strip_accents(str(text)).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def ieee_split_semicolon(value: str) -> List[str]:
    return [part.strip() for part in str(value or "").split(";") if part.strip()]


def ieee_is_placeholder_affiliation(value: str) -> bool:
    text = ieee_norm_for_matching(value)
    if not text:
        return False
    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in IEEE_PLACEHOLDER_PATTERNS)


def ieee_is_unam_affiliation(value: str) -> bool:
    text_plain = str(value or "")
    text_norm = ieee_norm_for_matching(text_plain)
    return any(re.search(pattern, text_plain, flags=re.IGNORECASE) or re.search(pattern, text_norm, flags=re.IGNORECASE) for pattern in IEEE_UNAM_PATTERNS)


def ieee_keep_two_affiliations_with_unam_priority(affiliations: List[str]) -> Tuple[str, str]:
    clean = []
    seen = set()
    for aff in affiliations:
        aff = str(aff or "").strip()
        if not aff or ieee_is_placeholder_affiliation(aff):
            continue
        key = ieee_norm_for_matching(aff)
        if key and key not in seen:
            clean.append(aff)
            seen.add(key)
    if len(clean) <= 2:
        return (clean[0] if len(clean) >= 1 else "", clean[1] if len(clean) >= 2 else "")
    unam = [aff for aff in clean if ieee_is_unam_affiliation(aff)]
    if unam:
        return (unam[0] if len(unam) >= 1 else "", unam[1] if len(unam) >= 2 else "")
    return clean[0], clean[1]


def process_ieee_rows(rows: List[Dict[str, str]], global_ids: List[int]) -> List[Dict[str, str]]:
    final_rows: List[Dict[str, str]] = []
    for row, article_id in zip(rows, global_ids):
        authors = ieee_split_semicolon(row.get("Autor_norm", ""))
        affs = ieee_split_semicolon(row.get("Afiliacion1", ""))
        n_authors = len(authors)
        n_affs = len(affs)
        if n_authors == n_affs and n_authors > 0:
            assigned_lists = [[affs[pos]] for pos in range(n_authors)]
        elif n_authors == 1 and n_affs >= 1:
            assigned_lists = [affs]
        elif n_authors > 1 and n_affs == 1:
            assigned_lists = [[affs[0]] for _ in authors]
        else:
            assigned_lists = [[] for _ in authors]
        for author_pos, author in enumerate(authors, start=1):
            candidates = assigned_lists[author_pos - 1] if author_pos - 1 < len(assigned_lists) else []
            af1, af2 = ieee_keep_two_affiliations_with_unam_priority(candidates)
            out = {col: row.get(col, "") for col in CANONICAL_COLUMNS}
            out["indice"] = str(article_id)
            out["Autor_norm"] = author
            out["Afiliacion1"] = af1
            out["Afiliacion2"] = af2
            final_rows.append(out)
    return final_rows

## Reglas PROQUEST

In [6]:
# -----------------------------------------------------------------------------
# PROQUEST
# -----------------------------------------------------------------------------

PROQUEST_PLACEHOLDER_RE = re.compile(
    r"(?i)(afiliaci[oó]n\s+(externa|no\s+unam|no\s+recuperada)|"
    r"external\s+affiliation|affiliation\s+not\s+(available|exported|retrieved)|"
    r"no\s+unam\s+en\s+la\s+fuente\s+consultada|sin\s+afiliaci[oó]n)"
)
PROQUEST_UNAM_TERMS = [
    "unam", "universidad nacional autonoma de mexico", "universidad nacional autónoma de méxico",
    "national autonomous university of mexico", "national autonomous university of méxico", "national university of mexico",
    "grid:grid.9486.3", "grid.9486.3", "ror.org/01tmp8f25", "iimas",
    "instituto de investigaciones en matematicas aplicadas y en sistemas", "instituto de investigaciones en matemáticas aplicadas y en sistemas",
    "facultad de estudios superiores", "fes iztacala", "fes cuautitlan", "fes cuautitlán",
    "instituto de matematicas", "instituto de matemáticas", "instituto de geofisica", "instituto de geofísica",
    "instituto de ingenieria", "instituto de ingeniería", "instituto de neurobiologia", "instituto de neurobiología",
    "instituto de astronomia", "instituto de astronomía", "centro de fisica aplicada y tecnologia avanzada",
    "centro de física aplicada y tecnología avanzada", "escuela nacional de estudios superiores",
]


def proquest_strip_accents(text: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", str(text or "")) if not unicodedata.combining(c))


def proquest_norm_text(text: str) -> str:
    text = proquest_strip_accents(str(text or "")).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def proquest_is_placeholder(text: str) -> bool:
    return bool(PROQUEST_PLACEHOLDER_RE.search(str(text or "")))


def proquest_contains_unam(text: str) -> bool:
    n = proquest_norm_text(text)
    return any(proquest_norm_text(term) in n for term in PROQUEST_UNAM_TERMS)


def proquest_split_authors(author_cell: str) -> List[str]:
    return [a.strip() for a in str(author_cell or "").split(";") if a.strip()]


def proquest_split_blocks(aff1: str, aff2: str = "") -> List[str]:
    joined = " , ".join([x.strip() for x in [str(aff1 or ""), str(aff2 or "")] if x and x.strip()])
    if not joined.strip():
        return []
    blocks = [b.strip() for b in re.split(r"\s+,\s+", joined) if b.strip()]
    return [b for b in blocks if b and not proquest_is_placeholder(b)]


def proquest_fragment_is_email_or_tag(fragment: str) -> bool:
    f = proquest_norm_text(fragment)
    return bool(re.search(r"<\/?email>|@", f))


def proquest_split_author_affiliations(block: str) -> List[str]:
    block = str(block or "").strip()
    if not block or proquest_is_placeholder(block):
        return []
    if ";" not in block:
        return [block]
    raw_parts = [p.strip() for p in block.split(";") if p.strip()]
    if len(raw_parts) <= 1:
        return [block]
    parts: List[str] = []
    for part in raw_parts:
        if proquest_fragment_is_email_or_tag(part):
            if parts:
                parts[-1] = parts[-1] + "; " + part
            else:
                parts.append(part)
            continue
        if not proquest_is_placeholder(part):
            parts.append(part)
    deduped: List[str] = []
    seen = set()
    for p in parts:
        if p and p not in seen:
            deduped.append(p)
            seen.add(p)
    return deduped


def proquest_choose_affiliation_slots(parts: List[str]) -> Tuple[str, str]:
    parts = [p for p in parts if p and not proquest_is_placeholder(p)]
    if not parts:
        return "", ""
    if len(parts) == 1:
        return parts[0], ""
    if len(parts) == 2:
        return parts[0], parts[1]
    unam_parts = [p for p in parts if proquest_contains_unam(p)]
    if unam_parts:
        kept = unam_parts[:2]
        return kept[0], kept[1] if len(kept) > 1 else ""
    return "", ""


def proquest_classify_article(n_authors: int, n_blocks: int) -> str:
    if n_authors == 0:
        return "sin_autores"
    if n_blocks == 0:
        return "sin_afiliaciones"
    if n_authors == n_blocks:
        return "posicional"
    if n_authors == 1:
        return "autor_unico"
    if n_blocks == 1 and n_authors > 1:
        return "compartida"
    return "ambiguo"


def process_proquest_rows(rows: List[Dict[str, str]], global_ids: List[int]) -> List[Dict[str, str]]:
    final_rows: List[Dict[str, str]] = []
    for row, article_id in zip(rows, global_ids):
        authors = proquest_split_authors(row.get("Autor_norm", ""))
        blocks = proquest_split_blocks(row.get("Afiliacion1", ""), row.get("Afiliacion2", ""))
        n_authors = len(authors)
        n_blocks = len(blocks)
        article_rule = proquest_classify_article(n_authors, n_blocks)
        if not authors:
            authors = [""]
            n_authors = 1
        for author_pos, author in enumerate(authors, start=1):
            selected_block = ""
            if article_rule == "posicional":
                selected_block = blocks[author_pos - 1]
            elif article_rule == "autor_unico":
                selected_block = "; ".join(blocks)
            elif article_rule == "compartida":
                selected_block = blocks[0]
            parts = proquest_split_author_affiliations(selected_block)
            aff1, aff2 = proquest_choose_affiliation_slots(parts)
            out = {col: row.get(col, "") for col in CANONICAL_COLUMNS}
            out["indice"] = str(article_id)
            out["Autor_norm"] = author
            out["Afiliacion1"] = aff1
            out["Afiliacion2"] = aff2
            final_rows.append(out)
    return final_rows

## Reglas ScienceDirect

In [7]:
# -----------------------------------------------------------------------------
# ScienceDirect
# -----------------------------------------------------------------------------

SD_PARTICLES = {"de", "del", "da", "do", "dos", "das", "la", "las", "los", "van", "von", "y", "e"}
SD_PLACEHOLDER_PATTERNS_NORM = [
    r"afiliacion externa", r"afiliacion no unam", r"afiliacion no recuperada", r"no recuperada por la fuente",
    r"exacta no exportada", r"external affiliation", r"not available",
]
SD_UNAM_PATTERNS_NORM = [
    r"\bunam\b", r"universidad nacional autonoma de mexico", r"national autonomous university of mexico",
    r"national university of mexico", r"univ nacl autonoma mex",
]


def sd_normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text or ""))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def sd_is_placeholder(text: str) -> bool:
    n = sd_normalize_text(text)
    if not n:
        return False
    return any(re.search(pattern, n) for pattern in SD_PLACEHOLDER_PATTERNS_NORM)


def sd_contains_unam(text: str) -> bool:
    n = sd_normalize_text(text)
    if not n:
        return False
    return any(re.search(pattern, n) for pattern in SD_UNAM_PATTERNS_NORM)


def sd_split_authors(author_cell: str) -> List[str]:
    return [part.strip() for part in str(author_cell or "").split(";") if part.strip()]


def sd_split_blocks(affiliation_cell: str) -> List[str]:
    return [part.strip() for part in re.split(r"\s*\|\s*", str(affiliation_cell or "")) if part.strip()]


def sd_parse_author_affiliation_block(block: str) -> Tuple[str, str, str]:
    block = str(block or "").strip()
    match = re.search(r"\s+[—–]\s+", block)
    if match:
        return block[: match.start()].strip(), block[match.end():].strip(), "dash"
    if ":" in block:
        author_prefix, affiliation_text = block.split(":", 1)
        return author_prefix.strip(), affiliation_text.strip(), "colon"
    return "", block, "sin_separador"


def sd_canonical_author_to_display(author: str) -> str:
    author = str(author or "").strip()
    if "," in author:
        surname, given = [part.strip() for part in author.split(",", 1)]
        return f"{given} {surname}".strip()
    return author


def sd_token_set_for_name(name: str) -> set:
    return {tok for tok in sd_normalize_text(name).split() if tok and tok not in SD_PARTICLES}


def sd_validate_author_block(canonical_author: str, block_author: str) -> Tuple[str, float]:
    if not block_author:
        return "sin_prefijo", 0.0
    display_author = sd_canonical_author_to_display(canonical_author)
    if sd_normalize_text(display_author) == sd_normalize_text(block_author):
        return "exacta", 1.0
    a_tokens = sd_token_set_for_name(display_author)
    b_tokens = sd_token_set_for_name(block_author)
    if not a_tokens or not b_tokens:
        return "no_validada", 0.0
    overlap = a_tokens & b_tokens
    score = len(overlap) / max(1, min(len(a_tokens), len(b_tokens)))
    a_initials = {tok[0] for tok in a_tokens if tok}
    b_initials = {tok[0] for tok in b_tokens if tok}
    if overlap and (a_initials & b_initials):
        score = max(score, 0.75)
    score = round(min(score, 1.0), 3)
    return ("tokens_coinciden", score) if score >= 0.50 else ("no_validada", score)


def sd_split_multiple_affiliations(affiliation_text: str) -> List[str]:
    pieces = [part.strip() for part in re.split(r"\s*;\s*", str(affiliation_text or "")) if part.strip()]
    pieces = [piece for piece in pieces if not sd_is_placeholder(piece)]
    seen = set()
    result = []
    for piece in pieces:
        key = sd_normalize_text(piece)
        if key not in seen:
            seen.add(key)
            result.append(piece)
    return result


def sd_limit_affiliations_for_output(affiliations: List[str]) -> List[str]:
    affiliations = [aff for aff in affiliations if aff and not sd_is_placeholder(aff)]
    if len(affiliations) <= 2:
        return affiliations
    unam_affiliations = [aff for aff in affiliations if sd_contains_unam(aff)]
    if unam_affiliations:
        return unam_affiliations[:2]
    return []


def process_sciencedirect_rows(rows: List[Dict[str, str]], global_ids: List[int]) -> List[Dict[str, str]]:
    final_rows: List[Dict[str, str]] = []
    for row, article_id in zip(rows, global_ids):
        authors = sd_split_authors(row.get("Autor_norm", ""))
        blocks = sd_split_blocks(row.get("Afiliacion1", ""))
        n_authors = len(authors)
        n_blocks = len(blocks)
        parsed_blocks = [sd_parse_author_affiliation_block(block) for block in blocks]
        for pos, author in enumerate(authors, start=1):
            raw_affiliation_text = ""
            if n_authors == n_blocks and pos <= n_blocks:
                prefix, raw_affiliation_text, sep = parsed_blocks[pos - 1]
            else:
                best_idx = None
                best_score = 0.0
                for i, (candidate_prefix, _, _) in enumerate(parsed_blocks):
                    _, candidate_score = sd_validate_author_block(author, candidate_prefix)
                    if candidate_score > best_score:
                        best_idx = i
                        best_score = candidate_score
                if best_idx is not None and best_score >= 0.50:
                    _, raw_affiliation_text, _ = parsed_blocks[best_idx]
            affiliations = sd_split_multiple_affiliations(raw_affiliation_text)
            limited = sd_limit_affiliations_for_output(affiliations)
            aff1 = limited[0] if len(limited) >= 1 else ""
            aff2 = limited[1] if len(limited) >= 2 else ""
            if sd_is_placeholder(aff1):
                aff1 = ""
            if sd_is_placeholder(aff2):
                aff2 = ""
            final_row = {col: row.get(col, "") for col in CANONICAL_COLUMNS}
            final_row["indice"] = str(article_id)
            final_row["Autor_norm"] = author
            final_row["Afiliacion1"] = aff1
            final_row["Afiliacion2"] = aff2
            final_rows.append(final_row)
    return final_rows

## Reglas SCOPUS

In [8]:
# -----------------------------------------------------------------------------
# SCOPUS
# -----------------------------------------------------------------------------

SCOPUS_UNAM_RE = re.compile(
    r"(\bUNAM\b|Universidad\s+Nacional\s+Aut[oó]noma\s+de\s+M[eé]xico|"
    r"National\s+Autonomous\s+University\s+of\s+Mexico|"
    r"Nacional\s+Aut[oó]noma\s+de\s+M[eé]xico)",
    flags=re.IGNORECASE,
)
SCOPUS_AFF_STOPWORDS = {
    "de", "del", "la", "las", "los", "el", "and", "of", "the", "for", "in", "on",
    "city", "ciudad", "mexico", "méxico", "mx", "cdmx", "cd", "united", "states",
    "usa", "us", "department", "departamento", "faculty", "facultad", "school",
    "institute", "instituto", "university", "universidad", "center", "centre",
    "research", "division", "laboratory", "lab", "campus", "av", "avenida", "s", "n", "sn", "cp", "c", "u",
}


def scopus_strip_accents(text):
    return "".join(ch for ch in unicodedata.normalize("NFKD", text or "") if not unicodedata.combining(ch))


def scopus_norm_text(text):
    text = scopus_strip_accents(text or "").lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def scopus_author_key(author):
    return scopus_norm_text(author)


def scopus_split_authors(text):
    return [x.strip() for x in (text or "").split(";") if x.strip()]


def scopus_split_affiliations(text):
    return [x.strip() for x in (text or "").split(";") if x.strip()]


def scopus_is_unam_aff(aff):
    return bool(SCOPUS_UNAM_RE.search(aff or ""))


def scopus_dedupe_preserve(seq):
    seen = set(); out = []
    for item in seq:
        key = scopus_norm_text(item)
        if item and key not in seen:
            seen.add(key); out.append(item)
    return out


def scopus_meaningful_tokens(text):
    toks = set(scopus_norm_text(text).split())
    return {t for t in toks if len(t) >= 3 and t not in SCOPUS_AFF_STOPWORDS}


def scopus_aff_similarity(a, b):
    na, nb = scopus_norm_text(a), scopus_norm_text(b)
    if not na or not nb:
        return 0.0
    if na == nb:
        return 1.0
    shorter, longer = (na, nb) if len(na) <= len(nb) else (nb, na)
    if len(shorter) >= 35 and shorter in longer:
        return 0.94
    ta, tb = scopus_meaningful_tokens(a), scopus_meaningful_tokens(b)
    if not ta or not tb:
        return difflib.SequenceMatcher(None, na, nb).ratio() * 0.65
    inter = ta & tb
    union = ta | tb
    jacc = len(inter) / len(union)
    overlap = len(inter) / min(len(ta), len(tb))
    seq = difflib.SequenceMatcher(None, na, nb).ratio()
    score = max(jacc, 0.78 * overlap, 0.72 * seq)
    if scopus_is_unam_aff(a) and scopus_is_unam_aff(b):
        distinctive = {t for t in inter if t not in {"unam", "nacional", "autonoma", "autonomous", "national"}}
        if distinctive:
            score = max(score, 0.86)
        elif "unam" in na and "unam" in nb and abs(len(na) - len(nb)) < 80:
            score = max(score, 0.82)
    return min(score, 1.0)


def scopus_final_two_affiliations(candidates):
    candidates = scopus_dedupe_preserve([c for c in candidates if c])
    if len(candidates) <= 2:
        return (candidates[0] if len(candidates) >= 1 else ""), (candidates[1] if len(candidates) >= 2 else "")
    unam = [c for c in candidates if scopus_is_unam_aff(c)]
    if unam:
        kept = unam[:2]
        return (kept[0] if len(kept) >= 1 else ""), (kept[1] if len(kept) >= 2 else "")
    return "", ""


def scopus_direct_assignment_for_article(authors, affs):
    n_a, n_f = len(authors), len(affs)
    assignments = [[] for _ in authors]
    rules = [""] * n_a; confs = [""] * n_a; motivos = [""] * n_a
    if n_a == 0:
        return assignments, rules, confs, motivos, False
    if n_f == 0:
        for i in range(n_a):
            rules[i] = "sin_afiliaciones_en_registro"; confs[i] = "baja"; motivos[i] = "El registro no contiene afiliaciones para asignar."
        return assignments, rules, confs, motivos, True
    if n_a == 1:
        assignments[0] = affs[:]
        rules[0] = "scopus_autor_unico_todas_las_afiliaciones"; confs[0] = "alta"
        motivos[0] = "Un solo autor; todas las afiliaciones del artículo corresponden al único autor."
        return assignments, rules, confs, motivos, True
    if n_f == 1:
        for i in range(n_a):
            assignments[i] = [affs[0]]
            rules[i] = "scopus_unica_afiliacion_compartida"; confs[i] = "media"
            motivos[i] = "Varios autores y una sola afiliación institucional; se asignó como afiliación compartida."
        return assignments, rules, confs, motivos, True
    if n_a == n_f:
        for i in range(n_a):
            assignments[i] = [affs[i]]
            rules[i] = "scopus_posicional_n_autores_igual_n_afiliaciones"; confs[i] = "media-alta"
            motivos[i] = "n_autores == n_afiliaciones; se asignó por orden autor-afiliación solicitado."
        return assignments, rules, confs, motivos, True
    return assignments, rules, confs, motivos, False


def scopus_build_initial_assignments(rows):
    assignments_by_article = []
    known_affs = defaultdict(list)
    for row in rows:
        authors = scopus_split_authors(row.get("Autor_norm", ""))
        affs = scopus_split_affiliations(row.get("Afiliacion1", ""))
        cand, rules, confs, motivos, _ = scopus_direct_assignment_for_article(authors, affs)
        article_assignments = []
        for i, author in enumerate(authors):
            cands = cand[i] if i < len(cand) else []
            rule = rules[i] if i < len(rules) else "pendiente_busqueda_en_celdas"
            conf = confs[i] if i < len(confs) else "baja"
            motivo = motivos[i] if i < len(motivos) else "Varios autores y varias afiliaciones; se buscará evidencia dentro de la base/celdas."
            article_assignments.append({"candidates": scopus_dedupe_preserve(cands), "regla": rule, "confianza": conf, "motivo": motivo})
            if cands:
                k = scopus_author_key(author)
                for aff in cands:
                    if aff not in known_affs[k]:
                        known_affs[k].append(aff)
        assignments_by_article.append(article_assignments)
    return assignments_by_article, known_affs


def scopus_propagate_assignments(rows, assignments_by_article, known_affs, max_passes=4):
    for _ in range(max_passes):
        added_this_pass = 0
        for article_idx, row in enumerate(rows):
            authors = scopus_split_authors(row.get("Autor_norm", ""))
            affs = scopus_split_affiliations(row.get("Afiliacion1", ""))
            n_a, n_f = len(authors), len(affs)
            if not (n_a > 1 and n_f > 1 and n_a != n_f):
                continue
            unam_affs = [aff for aff in affs if scopus_is_unam_aff(aff)]
            unique_unam_aff = unam_affs[0] if len(unam_affs) == 1 else ""
            for i, author in enumerate(authors):
                rec = assignments_by_article[article_idx][i]
                if rec["candidates"]:
                    continue
                known = known_affs.get(scopus_author_key(author), [])
                if not known:
                    continue
                matches = []
                for aff in affs:
                    best = max((scopus_aff_similarity(aff, known_aff) for known_aff in known), default=0.0)
                    if best >= 0.84:
                        matches.append(aff)
                if matches:
                    pass
                elif unique_unam_aff and any(scopus_is_unam_aff(a) for a in known):
                    matches = [unique_unam_aff]
                else:
                    continue
                rec["candidates"] = scopus_dedupe_preserve(matches)
                for m in matches:
                    k = scopus_author_key(author)
                    if m not in known_affs[k]:
                        known_affs[k].append(m); added_this_pass += 1
        if added_this_pass == 0:
            break


def scopus_apply_complete_fallback(rows, assignments_by_article):
    for article_idx, row in enumerate(rows):
        authors = scopus_split_authors(row.get("Autor_norm", ""))
        affs = scopus_split_affiliations(row.get("Afiliacion1", ""))
        n_a, n_f = len(authors), len(affs)
        if not (n_a > 1 and n_f > 1 and n_a != n_f):
            continue
        if n_a > n_f:
            q, r = divmod(n_a, n_f)
            group_sizes = [q + (1 if j < r else 0) for j in range(n_f)]
            author_to_affs = [[] for _ in authors]
            pos = 0
            for aff_idx, size in enumerate(group_sizes):
                for _ in range(size):
                    if pos < n_a:
                        author_to_affs[pos].append(affs[aff_idx]); pos += 1
        else:
            q, r = divmod(n_f, n_a)
            group_sizes = [q + (1 if i < r else 0) for i in range(n_a)]
            author_to_affs = [[] for _ in authors]
            pos = 0
            for author_idx, size in enumerate(group_sizes):
                for _ in range(size):
                    if pos < n_f:
                        author_to_affs[author_idx].append(affs[pos]); pos += 1
        for i in range(n_a):
            rec = assignments_by_article[article_idx][i]
            if rec["candidates"]:
                continue
            cands = scopus_dedupe_preserve(author_to_affs[i])
            if cands:
                rec["candidates"] = cands


def process_scopus_rows(rows: List[Dict[str, str]], global_ids: List[int]) -> List[Dict[str, str]]:
    assignments_by_article, known_affs = scopus_build_initial_assignments(rows)
    scopus_propagate_assignments(rows, assignments_by_article, known_affs)
    scopus_apply_complete_fallback(rows, assignments_by_article)
    final_rows: List[Dict[str, str]] = []
    for article_pos, (row, article_id) in enumerate(zip(rows, global_ids), start=1):
        authors = scopus_split_authors(row.get("Autor_norm", ""))
        for i, author in enumerate(authors, start=1):
            rec = assignments_by_article[article_pos - 1][i - 1]
            candidates = rec["candidates"]
            aff1, aff2 = scopus_final_two_affiliations(candidates)
            final = {col: row.get(col, "") for col in CANONICAL_COLUMNS}
            final["indice"] = str(article_id)
            final["Autor_norm"] = author
            final["Afiliacion1"] = aff1
            final["Afiliacion2"] = aff2
            final_rows.append(final)
    return final_rows

## Reglas WOS

In [9]:
# -----------------------------------------------------------------------------
# WOS
# -----------------------------------------------------------------------------

def wos_norm_text(value: str) -> str:
    value = value or ""
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = value.lower()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def wos_split_authors(value: str) -> List[str]:
    return [part.strip() for part in (value or "").split(";") if part.strip()]


def wos_dedupe_preserve_order(values: Iterable[str]) -> List[str]:
    seen = set()
    out: List[str] = []
    for value in values:
        value = (value or "").strip()
        if not value:
            continue
        key = wos_norm_text(value)
        if key not in seen:
            seen.add(key)
            out.append(value)
    return out


def wos_parse_bracket_blocks(value: str) -> List[Tuple[List[str], str]]:
    value = value or ""
    if "[" not in value or "]" not in value:
        return []
    pattern = re.compile(r"\[([^\]]+)\]\s*(.*?)(?=(?:\s*;\s*\[[^\]]+\]\s*)|$)", re.DOTALL)
    blocks: List[Tuple[List[str], str]] = []
    for match in pattern.finditer(value):
        authors_blob = match.group(1).strip()
        aff = match.group(2).strip().strip(";").strip()
        authors = wos_split_authors(authors_blob)
        if authors and aff:
            blocks.append((authors, aff))
    return blocks


def wos_split_affiliations_text(value: str) -> List[str]:
    blocks = wos_parse_bracket_blocks(value)
    if blocks:
        return [aff for _, aff in blocks]
    return [part.strip() for part in (value or "").split(";") if part.strip()]


def wos_extract_affiliations(row: Dict[str, str]) -> List[str]:
    affs = []
    for col in ("Afiliacion1", "Afiliacion2"):
        value = row.get(col, "") or ""
        affs.extend(wos_split_affiliations_text(value))
    return [aff.strip() for aff in affs if aff.strip()]


def wos_is_unam_affiliation(value: str) -> bool:
    n = wos_norm_text(value)
    return any(pattern in n for pattern in [
        "universidad nacional autonoma de mexico", "univ nacl autonoma mex", "univ nac autonoma mex",
        "national autonomous university of mexico", "unam",
    ])


def wos_aff_match(a: str, b: str) -> bool:
    na, nb = wos_norm_text(a), wos_norm_text(b)
    if not na or not nb:
        return False
    if na == nb:
        return True
    if len(na) >= 12 and len(nb) >= 12 and (na in nb or nb in na):
        return True
    if wos_is_unam_affiliation(a) and wos_is_unam_affiliation(b):
        return True
    return difflib.SequenceMatcher(None, na, nb).ratio() >= 0.94


def wos_add_to_memory(memory: Dict[str, List[str]], author: str, affiliations: Iterable[str]) -> None:
    key = wos_norm_text(author)
    if not key:
        return
    memory[key] = wos_dedupe_preserve_order(memory[key] + list(affiliations))


def wos_explicit_block_assignment(authors: List[str], blocks: List[Tuple[List[str], str]]) -> Dict[int, List[str]]:
    assignments: Dict[int, List[str]] = defaultdict(list)
    author_keys = [wos_norm_text(author) for author in authors]
    for block_authors, aff in blocks:
        for block_author in block_authors:
            block_key = wos_norm_text(block_author)
            for i, article_key in enumerate(author_keys):
                if block_key == article_key:
                    assignments[i].append(aff)
                    break
    return {i: wos_dedupe_preserve_order(vals) for i, vals in assignments.items()}


def wos_initial_assignments(row: Dict[str, str]) -> Tuple[Dict[int, List[str]], Dict[int, str], Dict[int, str], Dict[int, str]]:
    authors = wos_split_authors(row.get("Autor_norm", ""))
    affs = wos_extract_affiliations(row)
    blocks = wos_parse_bracket_blocks(row.get("Afiliacion1", "")) or wos_parse_bracket_blocks(row.get("Afiliacion2", ""))
    n_authors, n_affs = len(authors), len(affs)
    assignments: Dict[int, List[str]] = {}
    rules: Dict[int, str] = {}
    confidences: Dict[int, str] = {}
    motives: Dict[int, str] = {}
    if blocks:
        explicit = wos_explicit_block_assignment(authors, blocks)
        for i, vals in explicit.items():
            assignments[i] = vals
            rules[i] = "wos_marcador_autor_en_afiliacion"
            confidences[i] = "alta"
            motives[i] = "Afiliacion1/Afiliacion2 contiene mapeo explícito tipo [Autor; Autor] Afiliación."
        return assignments, rules, confidences, motives
    if n_authors == 1 and n_affs > 0:
        assignments[0] = affs[:]
        rules[0] = "wos_autor_unico"
        confidences[0] = "alta"
        motives[0] = "Un solo autor; se asignaron las afiliaciones del registro."
        return assignments, rules, confidences, motives
    if n_authors > 1 and n_authors == n_affs and n_affs > 0:
        for i, aff in enumerate(affs):
            assignments[i] = [aff]
            rules[i] = "wos_posicional_n_autores_igual_n_afiliaciones"
            confidences[i] = "media"
            motives[i] = "n_autores == n_afiliaciones; asignación por orden autor→afiliación solicitada."
        return assignments, rules, confidences, motives
    if n_authors > 1 and n_affs == 1:
        for i in range(n_authors):
            assignments[i] = [affs[0]]
            rules[i] = "wos_unica_afiliacion_articulo_para_todos"
            confidences[i] = "media"
            motives[i] = "Varios autores y una sola afiliación; se asignó la misma afiliación a todos."
        return assignments, rules, confidences, motives
    return assignments, rules, confidences, motives


def wos_build_author_memory(rows: List[Dict[str, str]]) -> Dict[str, List[str]]:
    memory: Dict[str, List[str]] = defaultdict(list)
    for row in rows:
        authors = wos_split_authors(row.get("Autor_norm", ""))
        assignments, _, _, _ = wos_initial_assignments(row)
        for i, affs in assignments.items():
            if 0 <= i < len(authors):
                wos_add_to_memory(memory, authors[i], affs)
    return memory


def wos_fallback_by_order(authors: List[str], affs: List[str]) -> Tuple[Dict[int, List[str]], str, str, str]:
    n, m = len(authors), len(affs)
    assignments: Dict[int, List[str]] = {}
    if n == 0 or m == 0:
        return assignments, "wos_sin_afiliaciones", "baja", "No hay afiliaciones disponibles en el registro."
    if n == 1:
        assignments[0] = affs[:]
        return assignments, "wos_autor_unico_fallback", "media", "Un solo autor; se asignaron las afiliaciones disponibles."
    if m == 1:
        for i in range(n):
            assignments[i] = [affs[0]]
        return assignments, "wos_unica_afiliacion_articulo_para_todos", "media", "Varios autores y una sola afiliación; se asignó a todos."
    if m > n:
        for i in range(n):
            start = round(i * m / n)
            end = round((i + 1) * m / n)
            if end <= start:
                end = min(start + 1, m)
            assignments[i] = affs[start:end]
        return assignments, "wos_fallback_afiliaciones_en_bloques_por_orden", "baja", "WOS no da mapeo explícito; se distribuyeron afiliaciones en bloques contiguos según el orden de autores."
    for i in range(n):
        aff_idx = int(i * m / n)
        aff_idx = min(max(aff_idx, 0), m - 1)
        assignments[i] = [affs[aff_idx]]
    return assignments, "wos_fallback_autores_en_bloques_por_orden", "baja", "WOS no da mapeo explícito; se distribuyeron autores en bloques contiguos según el orden de afiliaciones."


def wos_propagate_from_memory(author: str, affs: List[str], memory: Dict[str, List[str]]) -> Tuple[List[str], str, str, str]:
    previous = memory.get(wos_norm_text(author), [])
    if not previous:
        return [], "", "", ""
    matches = []
    for aff in affs:
        if any(wos_aff_match(aff, prev) for prev in previous):
            matches.append(aff)
    matches = wos_dedupe_preserve_order(matches)
    if matches:
        return matches, "wos_propagacion_autor_afiliacion_coincidente", "media", "El autor aparece en la base con una afiliación coincidente o muy similar dentro de las afiliaciones del artículo."
    article_unam = [aff for aff in affs if wos_is_unam_affiliation(aff)]
    if article_unam and any(wos_is_unam_affiliation(prev) for prev in previous):
        return wos_dedupe_preserve_order(article_unam), "wos_propagacion_unam_autor", "media", "El autor aparece asociado a UNAM en la base y el artículo contiene afiliación UNAM."
    return [], "", "", ""


def wos_finalize_affiliations(candidates: List[str], mode: str = "complete") -> Tuple[str, str]:
    unique = wos_dedupe_preserve_order(candidates)
    if len(unique) == 0:
        return "", ""
    if len(unique) == 1:
        return unique[0], ""
    if len(unique) == 2:
        return unique[0], unique[1]
    unam = wos_dedupe_preserve_order([aff for aff in unique if wos_is_unam_affiliation(aff)])
    if unam:
        aff1 = unam[0] if len(unam) >= 1 else ""
        aff2 = unam[1] if len(unam) >= 2 else ""
        return aff1, aff2
    if mode == "strict":
        return "", ""
    return unique[0], unique[1]


def process_wos_rows(rows: List[Dict[str, str]], global_ids: List[int]) -> List[Dict[str, str]]:
    # Se usa la modalidad completa, que es la base deseada para no dejar filas vacías cuando hay afiliaciones disponibles.
    memory = wos_build_author_memory(rows)
    final_rows: List[Dict[str, str]] = []
    for row, article_id in zip(rows, global_ids):
        authors = wos_split_authors(row.get("Autor_norm", ""))
        affs = wos_extract_affiliations(row)
        initial, init_rules, init_conf, init_motives = wos_initial_assignments(row)
        fallback, fallback_rule, fallback_conf, fallback_motive = wos_fallback_by_order(authors, affs)
        for pos0, author in enumerate(authors):
            if pos0 in initial and initial[pos0]:
                candidates = initial[pos0]
            else:
                candidates, rule, confidence, motive = wos_propagate_from_memory(author, affs, memory)
                if not candidates:
                    candidates = fallback.get(pos0, [])
            aff1, aff2 = wos_finalize_affiliations(candidates, mode="complete")
            out = {col: row.get(col, "") for col in CANONICAL_COLUMNS}
            out["indice"] = str(article_id)
            out["Autor_norm"] = author
            out["Afiliacion1"] = aff1
            out["Afiliacion2"] = aff2
            final_rows.append(out)
    return final_rows

## Ejecución integrada

In [10]:
# -----------------------------------------------------------------------------
# Procesador integrado explícito
# -----------------------------------------------------------------------------

def process_integrated_explicit(input_path: Path = INPUT_INTEGRADO, output_path: Path = OUTPUT_FINAL) -> Dict[str, Any]:
    rows = read_csv(input_path)
    grouped = group_integrated_rows(rows)
    final_rows_all: List[Dict[str, str]] = []

    processors = {
        "ACM": process_acm_rows,
        "EBSCO": process_ebsco_rows,
        "ENGINEERING_VILLAGE": process_ev_rows,
        "IEEE": process_ieee_rows,
        "PROQUEST": process_proquest_rows,
        "SCIENCEDIRECT": process_sciencedirect_rows,
        "SCOPUS": process_scopus_rows,
        "WOS": process_wos_rows,
    }

    source_counts = {}
    for source in SOURCE_ORDER:
        pairs = grouped[source]
        source_rows = [row for _, row in pairs]
        global_ids = [gidx for gidx, _ in pairs]
        if not source_rows:
            source_counts[source] = {"articulos": 0, "filas_autor": 0}
            continue
        generated = processors[source](source_rows, global_ids)
        for seq, row in enumerate(generated):
            row["__source"] = source
            row["__seq_source"] = str(seq)
        final_rows_all.extend(generated)
        source_counts[source] = {"articulos": len(source_rows), "filas_autor": len(generated)}

    final_rows_all.sort(
        key=lambda r: (
            int(r["indice"]) if str(r["indice"]).isdigit() else 10**12,
            SOURCE_ORDER.index(r["__source"]),
            int(r["__seq_source"]),
        )
    )
    final_for_output = [{col: row.get(col, "") for col in CANONICAL_COLUMNS} for row in final_rows_all]
    write_csv(output_path, final_for_output, CANONICAL_COLUMNS)

    with_aff = sum(1 for row in final_for_output if str(row.get("Afiliacion1", "")).strip() or str(row.get("Afiliacion2", "")).strip())
    aff2_rows = sum(1 for row in final_for_output if str(row.get("Afiliacion2", "")).strip())
    resumen = {
        "archivo_entrada": str(input_path),
        "archivo_salida": str(output_path),
        "articulos_originales": len(rows),
        "filas_finales_autor_articulo": len(final_for_output),
        "articulos_finales_unicos": len({row["indice"] for row in final_for_output}),
        "filas_con_alguna_afiliacion": with_aff,
        "filas_sin_afiliacion": len(final_for_output) - with_aff,
        "filas_con_Afiliacion2": aff2_rows,
        "articulos_por_fuente": {source: len(grouped[source]) for source in SOURCE_ORDER},
        "filas_autor_por_fuente": {source: source_counts[source]["filas_autor"] for source in SOURCE_ORDER},
    }
    return resumen


def main() -> None:
    resumen = process_integrated_explicit()
    print(json.dumps(resumen, ensure_ascii=False, indent=2))

if __name__ == "__main__":
    main()

{
  "archivo_entrada": "C:\\Users\\hazar\\Desktop\\PROYECTO\\02_modelo canonico\\03_union\\integrado_preliminar_canonico.csv",
  "archivo_salida": "C:\\Users\\hazar\\Desktop\\PROYECTO\\04_Limpieza\\00_Separacion_Autor_Afiliacion\\integrado_articulo_autor_afiliacion_unificado.csv",
  "articulos_originales": 1471,
  "filas_finales_autor_articulo": 8940,
  "articulos_finales_unicos": 1471,
  "filas_con_alguna_afiliacion": 8938,
  "filas_sin_afiliacion": 2,
  "filas_con_Afiliacion2": 268,
  "articulos_por_fuente": {
    "ACM": 57,
    "EBSCO": 129,
    "ENGINEERING_VILLAGE": 142,
    "IEEE": 158,
    "PROQUEST": 19,
    "SCIENCEDIRECT": 43,
    "SCOPUS": 561,
    "WOS": 362
  },
  "filas_autor_por_fuente": {
    "ACM": 191,
    "EBSCO": 498,
    "ENGINEERING_VILLAGE": 767,
    "IEEE": 960,
    "PROQUEST": 94,
    "SCIENCEDIRECT": 188,
    "SCOPUS": 4296,
    "WOS": 1946
  }
}
